<a href="https://colab.research.google.com/github/dougyd92/ML-Foudations/blob/main/Notebooks/11_ML_Workflow_Spot_the_Error.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 11: ML Best Practices & End-to-End Workflow
## Companion Notebook: "Spot the Error"

This notebook contains a **complete ML workflow** on the California Housing dataset. It runs without errors and produces metrics.

**But there are problems.**

The workflow contains several deliberate mistakes spanning the full pipeline. Your job is to find them, understand why they're wrong, and fix them.

### How This Works

1. **Sections 1–6** contain the "buggy" workflow. Read through it, run it, and identify the issues.
2. **Section 7** is where you'll write corrected code in the exercises.
3. A **scoreboard** at the end compares the original metrics to your improved metrics.

> **Important:** Save a copy of this notebook before making any changes!
> Go to **File → Save a copy in Drive** (or download it from GitHub).

In [ ]:
# ============================================================
# Setup — Do not modify this cell
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV,
    RandomizedSearchCV, KFold
)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("All imports successful!")

---
# Section 1: Problem Definition

We'll use the California Housing dataset to build a regression model.

In [ ]:
# Load the data
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target

print(f"Dataset shape: {df.shape}")
df.head()

---
# Section 2: Exploratory Data Analysis

Let's take a quick look at the data.

In [ ]:
df.describe()

Looks good. Let's move on to preprocessing.

---
# Section 3: Data Preparation

We'll scale the features for better performance. We'll also create a feature to capture the market segment of each district.

In [ ]:
# Create a market segment feature based on home values
def categorize_value(val):
    if val < 1.5:
        return "low"
    elif val < 3.0:
        return "medium"
    elif val < 4.5:
        return "high"
    else:
        return "very_high"

df['value_segment'] = df['MedHouseVal'].apply(categorize_value)

# One-hot encode the market segment
segment_dummies = pd.get_dummies(df['value_segment'], prefix='segment', drop_first=True)
df = pd.concat([df, segment_dummies], axis=1)
df = df.drop(columns=['value_segment'])

print(f"Added market segment features: {[c for c in df.columns if 'segment' in c]}")
print(f"New shape: {df.shape}")

In [ ]:
# Separate features and target
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

---
# Section 4: Model Selection & Training

DecisionTree seems like a good model for regression problems.

In [ ]:
# Train a decision tree
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

---
# Section 5: Evaluation

Let's see how the model performed.

In [ ]:
# Evaluate on test set
y_pred = tree.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Test RMSE: {rmse:.4f}")

The model achieved a low RMSE. This is a good result.

Let's see which features were useful for the model.

In [ ]:
importances = pd.Series(tree.feature_importances_, index=X.columns)
print("Feature importances:")
print(importances.sort_values(ascending=False))

---
# Section 6: Scoreboard

Let's record the metrics from this workflow. We'll compare these to the improved version later.

In [ ]:
# ============================================================
# Scoreboard — Buggy Workflow
# ============================================================

buggy_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
buggy_mae = mean_absolute_error(y_test, y_pred)
buggy_r2 = r2_score(y_test, y_pred)

# Also compute training metrics for later comparison
y_train_pred = tree.predict(X_train)
buggy_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))

print("=" * 50)
print("BUGGY WORKFLOW METRICS")
print("=" * 50)
print(f"  Train RMSE:  {buggy_train_rmse:.4f}")
print(f"  Test RMSE:   {buggy_rmse:.4f}")
print(f"  Test MAE:    {buggy_mae:.4f}")
print(f"  Test R²:     {buggy_r2:.4f}")
print("=" * 50)

---
---

# Section 7: Fix It!

Now let's go back through the workflow and fix the issues. Each exercise corresponds to one section of the buggy workflow above.

We'll reload the data fresh so our fixes start from a clean slate.

In [ ]:
# Reload the data fresh
housing = fetch_california_housing()
df_fix = pd.DataFrame(housing.data, columns=housing.feature_names)
df_fix['MedHouseVal'] = housing.target

print(f"Fresh data loaded: {df_fix.shape}")
print(f"\nFeatures: {list(housing.feature_names)}")
print(f"Target: MedHouseVal (median house value in $100,000s)")

---
## Exercise 1: Problem Definition

The buggy notebook jumped straight into loading data with no discussion of what we're doing or why.

### Task 1: Write a Problem Definition

In the markdown cell below, write a proper problem definition that includes:
- What are we predicting? (include units)
- What type of ML problem is this?
- What metric will we use to measure success?
- What is the baseline to beat?
- What would a useful model look like?

*Write your text here.*
(Double-click this cell to edit it.)

In [ ]:
#@title Click to reveal solution

# This is a markdown exercise, so here's what a good problem definition looks like.
# You would write this as a markdown cell in your own notebook.

problem_definition = """
SAMPLE PROBLEM DEFINITION:

Problem: Predict the median house value for California districts based on
census-derived features.

Target: MedHouseVal — median house value in units of $100,000.
A prediction of 2.5 means $250,000.

Problem type: Regression (continuous target variable).

Success metric: RMSE (Root Mean Squared Error), in the same units as the target.
An RMSE of 0.5 means our predictions are off by about $50,000 on average (penalizing
larger errors more heavily). We'll also report MAE and R² for a fuller picture.

Baseline: Predicting the mean house value for every district. Any useful model
must beat this baseline.

Stakeholder context: A model with RMSE under $50,000–$60,000 (0.5–0.6 in target units)
would be practically useful for rough district-level valuation.
"""
print(problem_definition)

---
## Exercise 2: Exploratory Data Analysis

The buggy notebook only ran `.describe()` and moved on. Let's do a proper investigation.

In [ ]:
df.describe()

What do you notice after taking a closer look?

### Task 1: Check for Missing Values

In [ ]:
# Check for missing values in each column
# Print the total number of missing values

# Your code here

In [ ]:
#@title Click to reveal solution

print("Missing values per column:")
print(df_fix.isnull().sum())
print(f"\nTotal missing values: {df_fix.isnull().sum().sum()}")

### Task 2: Visualize the Target Distribution

Plot a histogram of MedHouseVal. What do you notice?

In [ ]:
# Plot a histogram of the target variable
# Look for anything unusual

# Your code here

In [ ]:
#@title Click to reveal solution

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df_fix['MedHouseVal'], bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(df_fix['MedHouseVal'].mean(), color='coral', linestyle='--', linewidth=2,
           label=f"Mean: {df_fix['MedHouseVal'].mean():.2f}")
ax.set_xlabel('Median House Value ($100,000s)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Target Variable (MedHouseVal)')
ax.legend()
plt.tight_layout()
plt.show()

# Notice the spike at 5.0 — the target is capped!
capped = (df_fix['MedHouseVal'] == df_fix['MedHouseVal'].max()).sum()
print(f"\nValues at the cap ({df_fix['MedHouseVal'].max()}): {capped} "
      f"({capped / len(df_fix) * 100:.1f}% of data)")
print("These capped values represent districts where the true median value")
print("exceeds the measurement limit. This could affect model performance.")

### Task 3: Feature Correlations with Target

Compute the correlation of each feature with MedHouseVal. Which features are most predictive?

In [ ]:
# Compute correlations between features and the target
# Display them sorted by absolute value

# Your code here

In [ ]:
#@title Click to reveal solution

correlations = df_fix.corr()['MedHouseVal'].drop('MedHouseVal').sort_values(
    key=abs, ascending=False
)
print("Feature correlations with MedHouseVal:")
print("=" * 45)
for feat, corr in correlations.items():
    bar = "+" * int(abs(corr) * 30)
    sign = "+" if corr > 0 else "-"
    print(f"  {feat:12s}  {corr:+.3f}  {sign}{bar}")

### Task 4: Scatter Plots of Top Features vs. Target

Create scatter plots of the top 3 most correlated features against the target.

In [ ]:
# Create scatter plots for the top 3 features most correlated with the target
# Use alpha for transparency since we have ~20k points

# Your code here

In [ ]:
#@title Click to reveal solution

top_features = df_fix.corr()['MedHouseVal'].drop('MedHouseVal').abs().sort_values(
    ascending=False
).index[:3]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feat in zip(axes, top_features):
    ax.scatter(df_fix[feat], df_fix['MedHouseVal'], alpha=0.15, s=5, color='steelblue')
    ax.set_xlabel(feat)
    ax.set_ylabel('MedHouseVal')
    ax.set_title(f'{feat} vs. MedHouseVal')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Notice the horizontal line at MedHouseVal = 5.0 (the cap).")
print("Also look for nonlinear patterns — MedInc shows a clear curve.")

---
## Exercise 3: Data Preparation (Fixing Leakage)

The buggy notebook had multiple data preparation issues:
1. A **leaked feature** (`value_segment`) derived directly from the target
2. **Scaling before splitting** — test data leaked into the scaler
3. **Polynomial features applied before splitting** — same leakage problem
4. **No justification** for adding polynomial features

Let's fix all of these.

### Task 1: Identify and Remove the Leaked Feature

Look at the buggy notebook's Section 3. The `value_segment` feature was created from `MedHouseVal`. Why is this a problem?

Create clean X and y from `df_fix` (which has no leaked features).

In [ ]:
# Separate features and target from df_fix
# df_fix has the original data with no leaked features
# Hint: the original features are in housing.feature_names

# Your code here

In [ ]:
#@title Click to reveal solution

X_fix = df_fix[list(housing.feature_names)].copy()
y_fix = df_fix['MedHouseVal'].copy()

print(f"Features: {list(X_fix.columns)}")
print(f"Shape: {X_fix.shape}")
print(f"\nNo leaked features! 'value_segment' was derived from the target,")
print(f"so it would not be available at prediction time.")

### Task 2: Split First, Then Scale

Split the data into train and test sets BEFORE any preprocessing. Then fit the scaler on training data only.

In [ ]:
# 1. Split into train/test (80/20, random_state=42)
# 2. Fit StandardScaler on X_train only
# 3. Transform both X_train and X_test

# Your code here

In [ ]:
#@title Click to reveal solution

X_train_fix, X_test_fix, y_train_fix, y_test_fix = train_test_split(
    X_fix, y_fix, test_size=0.2, random_state=42
)

scaler_fix = StandardScaler()
X_train_scaled_fix = scaler_fix.fit_transform(X_train_fix)  # fit on train only
X_test_scaled_fix = scaler_fix.transform(X_test_fix)        # transform, NOT fit_transform

print(f"Train: {X_train_scaled_fix.shape}")
print(f"Test:  {X_test_scaled_fix.shape}")
print(f"\nScaler fit on training data only — no leakage!")

---
## Exercise 4: Model Selection & Training

The buggy notebook trained a single DecisionTree model with no baseline, no comparison, and no cross-validation. Let's do this properly.

### Task 1: Establish a Baseline

Compute the RMSE of always predicting the mean of `y_train`. Any useful model must beat this.

In [ ]:
# Compute the baseline RMSE: predict y_train.mean() for every test sample
# Hint: np.full(len(y_test_fix), y_train_fix.mean()) creates an array of the mean

# Your code here

In [ ]:
#@title Click to reveal solution

baseline_pred = np.full(len(y_test_fix), y_train_fix.mean())
baseline_rmse = np.sqrt(mean_squared_error(y_test_fix, baseline_pred))
baseline_mae = mean_absolute_error(y_test_fix, baseline_pred)

print(f"Baseline (predict mean = {y_train_fix.mean():.3f}):")
print(f"  RMSE: {baseline_rmse:.4f}  (${baseline_rmse * 100_000:,.0f})")
print(f"  MAE:  {baseline_mae:.4f}  (${baseline_mae * 100_000:,.0f})")
print(f"\nAny useful model must beat RMSE {baseline_rmse:.4f}")

### Task 2: Train and Compare Multiple Models

Train at least 3 different models using 5-fold cross-validation. Compare their performance.

Suggested models: Ridge, Random Forest, Gradient Boosting. But feel free to try others.

In [ ]:
# Train at least 3 models using cross_val_score with 5-fold CV
# Use scoring='neg_mean_squared_error' and convert to RMSE
# Print the mean and std of RMSE for each model
#
# Example pattern:
#   scores = cross_val_score(model, X_train_scaled_fix, y_train_fix,
#                            cv=5, scoring='neg_mean_squared_error')
#   rmse_scores = np.sqrt(-scores)

# Your code here

In [ ]:
#@title Click to reveal solution

models = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=1.0)': Ridge(alpha=1.0, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'KNN (k=5)': KNeighborsRegressor(n_neighbors=5),
}

print(f"{'Model':<25} {'Mean RMSE':>10} {'Std':>8}")
print("=" * 48)

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(
        model, X_train_scaled_fix, y_train_fix,
        cv=5, scoring='neg_mean_squared_error'
    )
    rmse_scores = np.sqrt(-scores)
    cv_results[name] = {'mean': rmse_scores.mean(), 'std': rmse_scores.std()}
    print(f"  {name:<23} {rmse_scores.mean():.4f}    {rmse_scores.std():.4f}")

print(f"\n  {'Baseline (mean)':<23} {baseline_rmse:.4f}")

### Task 3: Tune the Best Model

Take your best-performing model from Task 2 and tune its hyperparameters using GridSearchCV or RandomizedSearchCV.

In [ ]:
# Tune the best model from the comparison above
# Define a parameter grid and use GridSearchCV or RandomizedSearchCV
# Print the best parameters and best cross-validation score

# Your code here

In [ ]:
#@title Click to reveal solution

# Gradient Boosting was likely the best — let's tune it
# param_grid = {
#     'n_estimators': [100, 200, 300],
#     'max_depth': [3, 5, 7],
#     'learning_rate': [0.05, 0.1, 0.2],
# }

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [1, 2],
    'learning_rate': [0.1],
}

gb_search = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)
gb_search.fit(X_train_scaled_fix, y_train_fix)

best_rmse = np.sqrt(-gb_search.best_score_)
print(f"Best parameters: {gb_search.best_params_}")
print(f"Best CV RMSE:    {best_rmse:.4f}  (${best_rmse * 100_000:,.0f})")

---
## Exercise 5: Evaluation

The buggy notebook reported only test RMSE, declared it "good" with no justification, and performed no diagnostic analysis. Let's evaluate properly.

### Task 1: Report Multiple Metrics

Using your best tuned model, report RMSE, MAE, and R² on the test set. Also report training metrics to check for overfitting.

In [ ]:
# Get the best model from the grid search (or train your best model)
# Compute RMSE, MAE, R² on both training and test sets
# Print results clearly with labels

# Your code here

In [ ]:
#@title Click to reveal solution

best_model = gb_search.best_estimator_

# Training metrics
y_train_pred_fix = best_model.predict(X_train_scaled_fix)
train_rmse_fix = np.sqrt(mean_squared_error(y_train_fix, y_train_pred_fix))
train_mae_fix = mean_absolute_error(y_train_fix, y_train_pred_fix)
train_r2_fix = r2_score(y_train_fix, y_train_pred_fix)

# Test metrics
y_test_pred_fix = best_model.predict(X_test_scaled_fix)
test_rmse_fix = np.sqrt(mean_squared_error(y_test_fix, y_test_pred_fix))
test_mae_fix = mean_absolute_error(y_test_fix, y_test_pred_fix)
test_r2_fix = r2_score(y_test_fix, y_test_pred_fix)

print(f"{'Metric':<10} {'Train':>10} {'Test':>10}")
print("=" * 35)
print(f"  {'RMSE':<8} {train_rmse_fix:>10.4f} {test_rmse_fix:>10.4f}")
print(f"  {'MAE':<8} {train_mae_fix:>10.4f} {test_mae_fix:>10.4f}")
print(f"  {'R²':<8} {train_r2_fix:>10.4f} {test_r2_fix:>10.4f}")
print(f"\nBaseline RMSE: {baseline_rmse:.4f}")
print(f"\nIn dollar terms:")
print(f"  Test RMSE: ${test_rmse_fix * 100_000:,.0f}")
print(f"  Test MAE:  ${test_mae_fix * 100_000:,.0f}")

gap = train_rmse_fix - test_rmse_fix
if abs(gap) < 0.05:
    print(f"\nTrain/test gap is small ({gap:.4f}) — not much overfitting.")
else:
    print(f"\nTrain/test gap: {gap:.4f} — some overfitting present.")

### Task 2: Residual Analysis

Create a scatter plot of predicted vs. actual values, and a plot of residuals vs. predicted values. What patterns do you see?

In [ ]:
# Create two plots side by side:
# Left: Predicted vs Actual (with a diagonal reference line)
# Right: Residuals vs Predicted (with a horizontal line at 0)

# Your code here

In [ ]:
#@title Click to reveal solution

residuals = y_test_fix - y_test_pred_fix

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Predicted vs Actual
ax1.scatter(y_test_fix, y_test_pred_fix, alpha=0.2, s=5, color='steelblue')
ax1.plot([0, 5.5], [0, 5.5], 'k--', linewidth=1, label='Perfect prediction')
ax1.set_xlabel('Actual MedHouseVal')
ax1.set_ylabel('Predicted MedHouseVal')
ax1.set_title('Predicted vs. Actual')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Residuals vs Predicted
ax2.scatter(y_test_pred_fix, residuals, alpha=0.2, s=5, color='coral')
ax2.axhline(y=0, color='k', linestyle='--', linewidth=1)
ax2.set_xlabel('Predicted MedHouseVal')
ax2.set_ylabel('Residual (Actual - Predicted)')
ax2.set_title('Residuals vs. Predicted')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Things to look for:")
print("- Cluster of points at actual=5.0 (the capped values)")
print("- Whether residuals fan out at higher predictions (heteroscedasticity)")
print("- Any systematic patterns suggesting the model misses structure")

### Task 3: Interpret Your Results

Answer these questions in the markdown cell below:
1. How does your model compare to the baseline?
2. What does the RMSE mean in practical terms (dollar amounts)?
3. Where does the model struggle? (Look at the residual plot.)
4. What would you try next to improve performance?

*Your interpretation here...*

In [ ]:
#@title Click to reveal solution

# Sample interpretation:
interpretation = """
SAMPLE INTERPRETATION:

1. Baseline comparison: The mean-prediction baseline gives an RMSE of ~1.14
   ($114,000). Our tuned Gradient Boosting model achieves ~0.55 ($55,000),
   a significant improvement.

2. Practical meaning: On average, our predictions are off by about $55,000.
   For a $250,000 home, that's roughly a 21% error. Useful for rough
   district-level estimates, but not precise enough for individual appraisals.

3. Where it struggles: The residual plot shows the model consistently
   underpredicts for districts at the $500,000 cap. This makes sense — these
   are censored values where the true price could be much higher. The model
   also shows some heteroscedasticity (larger errors for higher-value districts).

4. Next steps:
   - Filter out capped values (MedHouseVal == 5.0) and retrain
   - Try feature engineering (e.g., rooms per household, population density)
   - Experiment with XGBoost or other advanced boosting methods
   - Consider log-transforming the target to handle right skew
"""
print(interpretation)

---
## Exercise 6: Final Scoreboard

Let's compare the buggy workflow to the fixed workflow side by side.

In [ ]:
# Run this cell after completing all exercises above.
# Make sure the buggy metrics (Section 6) have been computed by running those cells first.

# Collect fixed metrics from Exercise 5
# If you used the solution cells, the variables are already set.
# Otherwise, replace these with your own variable names.

try:
    print("=" * 60)
    print("BEFORE vs. AFTER COMPARISON")
    print("=" * 60)
    print(f"")
    print(f"{'Metric':<18} {'Buggy':>12} {'Fixed':>12} {'Change':>12}")
    print("-" * 60)
    print(f"  {'Test RMSE':<16} {buggy_rmse:>12.4f} {test_rmse_fix:>12.4f} "
          f"{test_rmse_fix - buggy_rmse:>+12.4f}")
    print(f"  {'Test MAE':<16} {buggy_mae:>12.4f} {test_mae_fix:>12.4f} "
          f"{test_mae_fix - buggy_mae:>+12.4f}")
    print(f"  {'Test R²':<16} {buggy_r2:>12.4f} {test_r2_fix:>12.4f} "
          f"{test_r2_fix - buggy_r2:>+12.4f}")
    print(f"  {'Train RMSE':<16} {buggy_train_rmse:>12.4f} {train_rmse_fix:>12.4f} "
          f"{train_rmse_fix - buggy_train_rmse:>+12.4f}")
    print(f"  {'Baseline RMSE':<16} {'':>12} {baseline_rmse:>12.4f}")
    print("-" * 60)
    print(f"")

    if test_rmse_fix > buggy_rmse:
        print("Wait — the fixed RMSE is HIGHER than the buggy RMSE?")
        print("")
        print("Yes! And that's the point.")
        print("")
        print("The buggy notebook's low RMSE was a lie. The 'value_segment'")
        print("feature was derived directly from the target variable, so the")
        print("model was essentially being told the answer. The preprocessing")
        print("leakage further inflated the results.")
        print("")
        print("The fixed RMSE is higher but HONEST. This model will actually")
        print("generalize to new data. The buggy model would fail in production.")
    else:
        print("Your fixed model beat the buggy model — nice work!")
        print("(This can happen depending on model choice and tuning.)")
    print("")
    print(f"Fixed model RMSE in dollar terms: ${test_rmse_fix * 100_000:,.0f}")
    print(f"Baseline RMSE in dollar terms:    ${baseline_rmse * 100_000:,.0f}")
    print(f"Improvement over baseline:        ${(baseline_rmse - test_rmse_fix) * 100_000:,.0f}")

except NameError as e:
    print(f"Missing variable: {e}")
    print("Make sure you've run both the buggy workflow (Sections 1-6)")
    print("and the fix-it exercises (Section 7) before running this cell.")

---
# Summary

**Key takeaways from this notebook:**

- **Problem definition matters.** Without clear goals and success criteria, you can't evaluate whether your model is useful.
- **EDA is an investigation, not a formality.** Plotting the target revealed the cap at 5.0; checking correlations showed which features to focus on.
- **Data leakage produces misleadingly good metrics.** The leaked feature and preprocessing leakage made the buggy model look great — but it was cheating.
- **"Worse" metrics can mean a better model.** Honest evaluation on properly prepared data is more valuable than inflated numbers.
- **Always establish a baseline.** Without one, you can't tell if your model is actually learning anything useful.
- **Compare multiple models and evaluate thoroughly.** One metric, one model, no diagnostics = no confidence in your results.

These are the same principles you'll apply in your **capstone project**. The workflow from this session — define, explore, prepare, model, evaluate — is your roadmap.